# 22_gemm

Audience: junior researcher

- Challenge path: `challenges/medium/22_gemm`
- Source spec: [challenges/medium/22_gemm/challenge.html](../challenge.html)
- Source implementation: [challenges/medium/22_gemm/challenge.py](../challenge.py)


## Problem Statement

The original challenge HTML is embedded below so the notebook stays close to the repo source.

<p>
    Implement a basic General Matrix Multiplication (GEMM). Given matrix \(A\) of dimensions \(M \times K\), matrix \(B\) of dimensions \(K \times N\), input/output matrix \(C\) of dimensions \(M \times N\), and scalar multipliers \( \alpha \) and \( \beta \), compute the operation:
    \[ C = \alpha \cdot (A \times B) + \beta \cdot C_{initial} \]
</p>
<p>
    The input matrices \(A\), \(B\), and the initial state of \(C\) contain 16-bit floating-point numbers (FP16/<code>half</code>). All matrices are stored in row-major order. The scalars \( \alpha \) and \( \beta \) are 32-bit floats.
</p>

<h2>Implementation Requirements</h2>
<ul>
    <li>Use only native features (external libraries other than WMMA are not permitted).</li>
    <li>The <code>solve</code> function signature must remain unchanged.</li>
    <li>Accumulation during multiplication should use FP32 for better precision before converting the final result to FP16.</li>
    <li>The final result must be stored back into matrix <code>C</code> as <code>half</code>.</li>
</ul>

<h2>Example:</h2>
<p>
Input:<br>
<em>(Note: Input matrices A, B, C_initial are FP16 type for the problem)</em><br>
Matrix \(A\) (\(M=2, K=3\)):
\[
\begin{bmatrix}
1.0 & 2.0 & 3.0 \\
4.0 & 5.0 & 6.0
\end{bmatrix}
\]
Matrix \(B\) (\(K=3, N=2\)):
\[
\begin{bmatrix}
1.0 & 2.0 \\
3.0 & 4.0 \\
5.0 & 6.0
\end{bmatrix}
\]
Matrix \(C_{initial}\) (\(M=2, N=2\)):
\[
\begin{bmatrix}
1.0 & 1.0 \\
1.0 & 1.0
\end{bmatrix}
\]
\[\alpha = 1.0 \text{ (FP32)}\]
\[\beta = 0.0 \text{ (FP32)}\]

Output (FP16):<br>
Matrix \(C\) (\(M=2, N=2\)):
\[
\begin{bmatrix}
22.0 & 28.0 \\
49.0 & 64.0
\end{bmatrix}
\]
</p>

<h2>Constraints</h2>
<ul>
    <li>16 &le; <code>M</code>, <code>N</code>, <code>K</code> &le; 4096</li>

  <li>Performance is measured with <code>K</code> = 1,024, <code>M</code> = 1,024, <code>N</code> = 1,024</li>
</ul>


## Framework Coverage

This notebook collects the currently available solution artifacts for PyTorch, JAX, Triton, and MLX.

## Pytorch

Source: `challenges/medium/22_gemm/solution/solution.pytorch.py`

In [ ]:
import torch


def solve(
    A: torch.Tensor,
    B: torch.Tensor,
    C: torch.Tensor,
    M: int,
    N: int,
    K: int,
    alpha: float,
    beta: float,
):
    A_f32 = A.view(M, K).to(torch.float32)
    B_f32 = B.view(K, N).to(torch.float32)
    C_f32 = C.view(M, N).to(torch.float32)
    C.copy_((alpha * (A_f32 @ B_f32) + beta * C_f32).to(C.dtype))


## Jax

Source: `challenges/medium/22_gemm/solution/solution.jax.py`

In [ ]:
import jax
import jax.numpy as jnp


@jax.jit
def solve(
    A: jax.Array, B: jax.Array, M: int, N: int, K: int, alpha: float, beta: float
) -> jax.Array:
    A_f32 = A.reshape(M, K).astype(jnp.float32)
    B_f32 = B.reshape(K, N).astype(jnp.float32)
    result = alpha * (A_f32 @ B_f32) + beta * jnp.zeros((M, N), dtype=jnp.float32)
    return result.astype(jnp.float16)


## Triton

Source: `challenges/medium/22_gemm/solution/solution.triton.py`

In [ ]:
import torch

try:
    import triton  # noqa: F401
    import triton.language as tl  # noqa: F401
except Exception:  # pragma: no cover
    triton = None
    tl = None


def solve(
    A: torch.Tensor,
    B: torch.Tensor,
    C: torch.Tensor,
    M: int,
    N: int,
    K: int,
    alpha: float,
    beta: float,
):
    A_f32 = A.view(M, K).to(torch.float32)
    B_f32 = B.view(K, N).to(torch.float32)
    C_f32 = C.view(M, N).to(torch.float32)
    C.copy_((alpha * (A_f32 @ B_f32) + beta * C_f32).to(C.dtype))


## Mlx

Source: `challenges/medium/22_gemm/solution/solution.mlx.py`

In [ ]:
def solve(A, B, M: int, N: int, K: int, alpha: float, beta: float):
    try:
        import mlx.core as mx

        A_f32 = mx.array(A).reshape(M, K).astype(mx.float32)
        B_f32 = mx.array(B).reshape(K, N).astype(mx.float32)
        result = alpha * mx.matmul(A_f32, B_f32)
        return result.astype(mx.float16)
    except Exception:
        import torch

        A_f32 = torch.as_tensor(A).reshape(M, K).to(torch.float32)
        B_f32 = torch.as_tensor(B).reshape(K, N).to(torch.float32)
        return (alpha * (A_f32 @ B_f32)).to(torch.float16)


## Verification Notes

Use `python scripts/verify_matrix_solutions.py` for the local matrix-operation verifier.
GPU-only Triton validation still depends on a remote NVIDIA environment.